### Imports

In [0]:
# ============================================================================
# SMART MANUFACTURING INTELLIGENCE PLATFORM (SMIP)
#
# Gold Layer
#
# Notebook : 04_traceability_summary
# ============================================================================

from pyspark.sql.functions import col

from framework.core.session import spark

from framework.core.configuration import (
    SILVER_LAYER,
    GOLD_LAYER
)

from framework.core.logger import (
    banner,
    info,
    success
)

from framework.io.delta import write_delta

In [0]:
banner("Gold Layer - Traceability Summary")

### Read Silver

In [0]:
production = (

    spark.table(f"{SILVER_LAYER}.fact_production")

    .select(

        "product_key",

        "serial_number",

        "execution_id",

        "work_order_id",

        "product_code",

        "product_name",

        "planned_shift",

        "manufacturing_date"

    )

)

press = (

    spark.table(f"{SILVER_LAYER}.fact_press_operations")

    .select(

        "execution_id",

        "serial_number",

        "machine_key",

        "operator_key",

        "tool_key",

        "factory_key",

        "press_operation_id",

        "operation_number",

        "operation_name",

        "target_force_kn",

        "actual_force_kn",

        "force_deviation_kn",

        "cycle_time_sec"

    )

)

quality = (

    spark.table(f"{SILVER_LAYER}.fact_quality")

    .select(

        "execution_id",

        "serial_number",

        "test_program_id",

        "test_name",

        "target_value",

        "measured_value",

        "unit",

        "result",

        "material_number",

        "batch_number",

        "supplier",

        "scan_status",

        "package_type",

        "package_weight_kg",

        "packaging_status"

    )

)

products = (

    spark.table(f"{SILVER_LAYER}.dim_products")

    .select(

        "product_key",

        "family"

    )

)

machines = (

    spark.table(f"{SILVER_LAYER}.dim_machines")

    .select(

        "machine_key",

        "machine_id",

        "machine_name",

        "machine_type"

    )

)

operators = (

    spark.table(f"{SILVER_LAYER}.dim_operators")

    .select(

        "operator_key",

        "operator_id",

        "operator_name",

        "skill_level"

    )

)

tools = (

    spark.table(f"{SILVER_LAYER}.dim_tools")

    .select(

        "tool_key",

        "tool_id",

        "tool_name",

        "tool_type"

    )

)

factory = (

    spark.table(f"{SILVER_LAYER}.dim_factory")

    .select(

        "factory_key",

        "hall_name",

        "line_name"

    )

)

info("Silver tables loaded.")

### Build Traceability

In [0]:
traceability = (

    production

    .join(

        press,

        [

            "execution_id",

            "serial_number"

        ],

        "left"

    )

    .join(

        quality,

        [

            "execution_id",

            "serial_number"

        ],

        "left"

    )

    .join(

        products,

        "product_key",

        "left"

    )

    .join(

        machines,

        "machine_key",

        "left"

    )

    .join(

        operators,

        "operator_key",

        "left"

    )

    .join(

        tools,

        "tool_key",

        "left"

    )

    .join(

        factory,

        "factory_key",

        "left"

    )

)

### Final Select

In [0]:
traceability = traceability.select(

    "serial_number",

    "execution_id",

    "work_order_id",

    "product_code",

    "product_name",

    "family",

    "planned_shift",

    "manufacturing_date",

    "hall_name",

    "line_name",

    "machine_id",

    "machine_name",

    "machine_type",

    "operator_id",

    "operator_name",

    "skill_level",

    "tool_id",

    "tool_name",

    "tool_type",

    "press_operation_id",

    "operation_number",

    "operation_name",

    "target_force_kn",

    "actual_force_kn",

    "force_deviation_kn",

    "cycle_time_sec",

    "test_program_id",

    "test_name",

    "target_value",

    "measured_value",

    "unit",

    "result",

    "material_number",

    "batch_number",

    "supplier",

    "scan_status",

    "package_type",

    "package_weight_kg",

    "packaging_status"

)

### Write Gold

In [0]:
write_delta(

    traceability,

    f"{GOLD_LAYER}.traceability_summary"

)

success("gold.traceability_summary created successfully.")

### Validation

In [0]:
display(traceability)

display(

    spark.sql(f"""

    SELECT

        COUNT(*) rows,

        COUNT(DISTINCT serial_number) serials

    FROM {GOLD_LAYER}.traceability_summary

    """)

)
